# 07 — Styling: Colours and Line Widths

A tour of the ways eucare can colour a tiling — from one-liner "just make it look interesting" calls to fine-grained per-element overrides. Pick whichever fits the picture you want to make.

The full menu, in order of increasing effort:

1. **Auto-colour at render time** — `G.show(face_color_by='congruency')` and friends. No mutation, one kwarg.
2. **Bake colours into the graph** — when you want the colours to stick (across renders, transformations, or files).
3. **Paint by hand** — write your own `face['color_key']` / `edge['color_key']` on faces and edges chosen by any predicate.
4. **Inherit through Conway operators** — propagate colours from an input graph onto the output of `chamfer`, `kis`, etc.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    shrink_rotate,
    rendering,
)
from eucare.rendering import multi_show


## Auto-colour at render time

The fastest path: pass a `face_color_by=` / `edge_color_by=` / `vertex_color_by=` kwarg to `G.show()` and you're done — no mutation, no extra imports.

The presets cover the common cases:

- **faces:** `"congruency"` (same shape), `"order"` (same side count)
- **edges:** `"length"`, `"orientation"` (angle mod π)
- **vertices:** `"order"` (degree)

You can also pass a `Classifier` instance or any callable `element → hashable` if the presets don't fit. Colours come from matplotlib's `tab10` by default; pass `face_cmap=`, `edge_cmap=` or `vertex_cmap=` to swap in `"viridis"`, `"plasma"`, `"Set2"`, a `Colormap` instance, or a hand-picked list of colours. The renderer figures out whether the cmap is qualitative (use the entries directly) or continuous (sample across the full gradient), and falls back to evenly-spaced `hsv` when a small qualitative cmap runs out of slots.

In [ ]:
def show_coloring_variants(G):
    multi_show(
        [G, G, G, G],
        titles=[
            'by face order (tab10)',
            'by face congruence (tab10)',
            'by face congruence (viridis)',
            'by edge orientation',
        ],
        render_faces=True, face_inset=0.05, render_vertices=False,
        per_subplot_kwargs=[
            dict(face_color_by='order', render_edges=False),
            dict(face_color_by='congruency', render_edges=False),
            dict(face_color_by='congruency', face_cmap='viridis', render_edges=False),
            dict(edge_color_by='orientation', render_faces=False, line_width="300%"),
        ],
    )

show_coloring_variants(example_graphs.from_tiles(example_tilesets.t_4_6_12(), rings=2))
show_coloring_variants(example_graphs.from_tiles(example_tilesets.t_4_6_12(), rings=2).gyro())



## Baking colours into the graph

Sometimes you want the colours to *stick* — to travel with the graph through a transformation, or to survive a save-and-load round-trip. `colorization.colorize(G, classifier)` writes a class id directly into each `face['color_key']`; the renderer then turns those ids into colours exactly like the `face_color_by=` path above.

A nice place to use this: colour a hyperbolic tiling by congruence *while it's still hyperbolic* (where lengths and angles actually match), then project to the Poincaré disk for display — the colours come along for the ride.

In [ ]:
from eucare.classifiers import congruency_classifier

# A hyperbolic {7, 3} *expand* tiling — heptagons, squares, and triangles.
G = example_graphs.from_tiles(example_tilesets.curved_expand(7, 3), rings=3)
print('geometry:', G.geometry.__name__)

# Bake congruency colours while the lengths and angles are still hyperbolic.
colorization.colorize(G, congruency_classifier())

# Project to the Poincaré disk — face['color_key'] travels with the graph.
G.convert_to_euclidean()
G.show(render_faces=True, face_inset=0.05, render_vertices=False)

## Paint by hand: face colours

For a one-off highlight, just set `face['color_key']` directly with whichever faces you want to call out. Anything that looks like a colour (`(r, g, b)`, `(r, g, b, a)`, or `"#rrggbb"`) is used as-is; everything else gets palette-mapped. Below: every face near the centre, painted orange.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G.recompute_lengths_and_angles()
for f in G.faces:
    if np.linalg.norm(f.midpoint()) < 1.5:
        f['color_key'] = (1.0, 0.6, 0.0, 0.9)  # opaque orange
G.show(render_faces=True, face_inset=0.05, render_vertices=False)


## Paint by hand: edges and widths

Edges work the same way — `edge['color_key']` for colour, `edge['line_width']` for stroke width. Half-edges come in directed pairs (`h` and `h.rev`), so set the attribute on both for consistent rendering. Below: the boundary of the central face, thick and red.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=3)
G.recompute_lengths_and_angles()
central = G.central_face()
for h in central.halfedge_iter():
    h['color_key'] = h.rev['color_key'] = (0.85, 0.10, 0.10, 1.0)
    h['line_width'] = h.rev['line_width'] = 0.2
G.show(face_inset=0.05, render_vertices=False)


## Inherit through Conway operators

Every Conway operator (`chamfer`, `kis`, `dual`, ...) tags some of the elements in its output with `obj['pre_conway']` → the element of the *input* graph they descended from. That gives you a one-line way to propagate any per-face attribute — colours included — through the transformation. Useful for "show what happened to this face" diagrams.

In [ ]:
from eucare.half import Face

G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=1)
central_face = G.central_face()
central_face['color_key'] = (1.0, 0.5, 0.1, 0.9)

for v in central_face.vertex_iter():
    v['color_key'] = np.random.rand(3)

D = conway.chamfer_graph()(G.copy(), delete_on_border=False)
D.recompute_lengths_and_angles()
for obj in D.vertices.union(D.faces).union(D.halfedges):
    src = obj.attributes.get('pre_conway')
    if isinstance(src, Face) and 'color_key' in src.attributes:
        obj['color_key'] = src['color_key']

multi_show([G, D],
           titles=['original', 'dual (colors inherited from original)'],
           render_faces=True, render_vertices=True, face_inset=0,
           vertex_radius=0.1)
